# 4 · Try your models on a new photo

Two saved models work together: the detector finds whole dominoes; the classifier
reads each half as 0–15. This notebook loads your existing checkpoints from Drive.
It does not train, change labels, or need the original dataset archive.

1. Run Setup. If it requests a restart, restart the session and run Setup again.
2. Check the run names and blank-half score below. A CPU runtime is enough.
3. Choose **Runtime → Run all**, mount Drive, then upload a JPEG or PNG when prompted.
4. Inspect the numbered outlines, predicted values, and straightened crops.
   Rerun **Upload and test** for another photo.

Try fresh arrangements and count the values yourself to compare. These are exploratory
predictions, not an accuracy measurement. Confident predictions can still miss or
double-count tiles; inspect every outline. Keep the frozen test set for final evaluation.

## Setup
Use the same pinned libraries as the training notebooks. No mobile export is needed.

In [1]:
"""Embedded in each notebook's setup cell; uses only the standard library."""
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version


NUMPY_REQUIREMENT = "numpy==2.1.3" if sys.version_info >= (3, 13) else "numpy==1.26.4"
TENSORFLOW_REQUIREMENT = "tensorflow==2.20.0" if sys.version_info >= (3, 13) else "tensorflow==2.19.1"
KERAS_REQUIREMENT = "keras==3.10.0" if sys.version_info >= (3, 13) else "keras==3.9.2"


def install_packages(requirements):
    if os.environ.get("DOMINO_SKIP_INSTALL") == "1":
        return
    if not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("Use a Python 3.10–3.13 runtime for these pinned packages.")
    if sys.platform == "darwin" and sys.version_info >= (3, 13):
        raise RuntimeError("Use Colab for Python 3.13, or Python 3.12 for local macOS runs.")
    restart_message = "Choose Runtime → Restart session, then rerun setup before continuing."
    if globals().get("_DOMINO_RESTART_REQUIRED"):
        raise RuntimeError(restart_message)

    def installed(name):
        try:
            return package_version(name)
        except PackageNotFoundError:
            return None

    modules = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
               "pillow": "PIL", "opencv-python": "cv2", "torch": "torch",
               "torchvision": "torchvision", "tensorflow": "tensorflow", "keras": "keras",
               "scipy": "scipy", "scikit-learn": "sklearn", "h5py": "h5py",
               "jax": "jax", "jaxlib": "jaxlib", "ml-dtypes": "ml_dtypes",
               "litert-torch": "litert_torch", "torchao": "torchao"}
    loaded = {name: installed(name) for name, module in modules.items() if module in sys.modules}
    print("Installing packages for Python", sys.version.split()[0], "— progress follows:", flush=True)
    command = [sys.executable, "-u", "-m", "pip", "install",
               "--only-binary=numpy,pandas", *requirements]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()

    # Colab can preload NumPy. Changing files on disk cannot replace its loaded extensions.
    changed = [name for name, previous in loaded.items() if installed(name) != previous]
    if changed:
        globals()["_DOMINO_RESTART_REQUIRED"] = True
        print("Loaded packages changed:", ", ".join(changed), flush=True)
    if returncode:
        raise RuntimeError("Package installation failed. Read the pip error above before continuing. "
                           + (restart_message if changed else ""))
    if changed:
        raise RuntimeError("Installation finished. " + restart_message)
    print("Installation finished. Continue to configuration.", flush=True)


install_packages([TENSORFLOW_REQUIREMENT, KERAS_REQUIREMENT, NUMPY_REQUIREMENT,
                  "pandas==2.2.3", "matplotlib==3.10.1", "pillow==11.1.0",
                  "opencv-python==4.11.0.86", "ultralytics==8.4.144",
                  "torch==2.9.0", "torchvision==0.24.0"])

Installing packages for Python 3.13.18 — progress follows:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.8/620.8 MB 31.0 MB/s  0:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 15.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 12.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 15.3 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 14.1 MB/s  0:00:42
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 12.0 MB/s  0:00:47
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 13.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 20.9 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RuntimeError: Installation finished. Choose Runtime → Restart session, then rerun setup before continuing.

## Choose your saved runs
The defaults use the original detector and your second classifier run.
`BLANK_SCORE` is the number of points for each blank half; use 0, 25, or 50.
Leave `PHOTO_PATH` empty to upload photos in Colab, or set it to a JPEG/PNG path.

In [ ]:
DETECTOR_RUN_NAME = "domino-v1-01"
CLASSIFIER_RUN_NAME = "pips-v1-02"
BLANK_SCORE = 50
DETECTION_CONFIDENCE = 0.25
PIP_CONFIDENCE = 0.80
PHOTO_PATH = ""
assert BLANK_SCORE in {0, 25, 50}
assert 0 < DETECTION_CONFIDENCE <= 1 and 0 < PIP_CONFIDENCE <= 1

In [ ]:
"""Shared helpers embedded into both standalone Colab notebooks by build_notebooks.py."""
from pathlib import Path
from collections import Counter
import csv
import hashlib
import io
import json
import math
import os
import random
import shutil
import sys
import zipfile

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

CLASS_NAMES = [str(i) for i in range(16)]
CROP_VERSION = "long-edge-rgb-128-v1"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def write_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_csv(path):
    with Path(path).open(newline="") as f:
        return list(csv.DictReader(f))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def setup_project():
    """Drive stores durable artifacts; Colab's local disk stores training input."""
    local_override = os.environ.get("DOMINO_ML_ROOT")
    in_colab = not local_override and "google.colab" in sys.modules
    if not local_override:
        try:
            import google.colab  # noqa: F401
            in_colab = True
        except ImportError:
            in_colab = False
    if in_colab:
        from google.colab import drive
        drive.mount("/content/drive")
        project = Path("/content/drive/MyDrive/DominoesCalculator")
        work = Path("/content/domino-work")
    else:
        if local_override:
            project = Path(local_override).expanduser().resolve()
        else:
            candidates = [Path.cwd(), *Path.cwd().parents]
            repo = next((p for p in candidates if (p / "machine_learning").is_dir()), None)
            if repo is None:
                raise RuntimeError("Set DOMINO_ML_ROOT to a local machine_learning folder.")
            project = repo / "machine_learning"
        work = Path(os.environ.get("DOMINO_WORK_DIR", str(project / "data/work")))
    project.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    return project, work, in_colab


def stage_export(project, work, dataset_name):
    """Accept an extracted export or a ZIP uploaded to the project folder."""
    source = project / "data/raw" / dataset_name
    if not source.is_dir():
        archive = project / f"{dataset_name}.zip"
        if not archive.is_file():
            raise FileNotFoundError(f"Upload {archive.name} to {project}, or place images/ and labels/ in {source}.")
        unpacked = work / "unpacked" / dataset_name
        if unpacked.exists():
            shutil.rmtree(unpacked)  # Only this generated local extraction cache.
        unpacked.mkdir(parents=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                target = (unpacked / member.filename).resolve()
                if not target.is_relative_to(unpacked.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Archive contains an unsafe path or symbolic link.")
            z.extractall(unpacked)
        candidates = [p.parent for p in unpacked.rglob("classes.txt")
                      if (p.parent / "images").is_dir() and (p.parent / "labels").is_dir()]
        if len(candidates) != 1:
            raise ValueError("Expected exactly one export with classes.txt, images/, and labels/.")
        source.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(candidates[0], source, ignore=shutil.ignore_patterns(".DS_Store", "__MACOSX"))
    # Refresh the generated local cache when source files change; never change the original export.
    local = work / "raw" / dataset_name
    shutil.copytree(source, local, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".DS_Store", "__MACOSX"))
    source_names = {p.relative_to(source) for p in source.rglob("*") if p.is_file() and p.name != ".DS_Store"}
    for p in local.rglob("*"):
        if p.is_file() and p.relative_to(local) not in source_names:
            p.unlink()
    return local


def read_rgb(path):
    # Match browser/phone display orientation before applying annotation coordinates.
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))


def audit_export(root):
    if (root / "classes.txt").read_text().splitlines() != ["domino"]:
        raise ValueError("This pipeline requires exactly one class: domino, ID 0.")
    images = sorted(p for p in (root / "images").iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = {p.stem: p for p in (root / "labels").glob("*.txt")}
    if not images or len({p.stem for p in images}) != len(images):
        raise ValueError("No supported images, or duplicate image stems.")
    if {p.stem for p in images} != set(labels):
        raise ValueError("Every image needs one matching label file; orphan labels are also rejected.")
    records = []
    for path in images:
        rgb = read_rgb(path)
        h, w = rgb.shape[:2]
        boxes = []
        for line_no, line in enumerate(labels[path.stem].read_text().splitlines(), 1):
            if not line.strip():
                continue
            fields = line.split()
            if len(fields) != 9 or fields[0] != "0":
                raise ValueError(f"{path.stem}:{line_no}: expected class 0 and eight OBB corner coordinates.")
            corners = np.asarray(fields[1:], dtype=np.float32).reshape(4, 2)
            if not np.isfinite(corners).all() or (corners < 0).any() or (corners > 1).any():
                raise ValueError(f"{path.stem}:{line_no}: coordinates must be finite and within [0, 1].")
            pixel_corners = corners * [w, h]
            if not cv2.isContourConvex(pixel_corners.astype(np.float32)) or abs(cv2.contourArea(pixel_corners.astype(np.float32))) < 4:
                raise ValueError(f"{path.stem}:{line_no}: invalid or tiny quadrilateral.")
            boxes.append(corners.tolist())
        records.append({"image": path.name, "stem": path.stem, "width": w, "height": h,
                        "image_sha256": sha256(path), "label_sha256": sha256(labels[path.stem]), "boxes": boxes})
    fingerprint = digest_json(records)
    print(f"Validated {len(records)} photos and {sum(len(r['boxes']) for r in records)} domino boxes.")
    return records, fingerprint


def contact_sheet(records, root, start=0, count=12, overlays=False):
    selected = records[start:start + count]
    if not selected:
        return
    fig, axes = plt.subplots(math.ceil(len(selected) / 3), 3, figsize=(15, 3.8 * math.ceil(len(selected) / 3)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, record in zip(axes.flat, selected):
        image = read_rgb(root / "images" / record["image"])
        ax.imshow(image)
        if overlays:
            for box in record["boxes"]:
                points = np.asarray(box) * [record["width"], record["height"]]
                points = np.vstack([points, points[0]])
                ax.plot(points[:, 0], points[:, 1], color="#f6ac48", linewidth=1)
        ax.set_title(record["image"], fontsize=9)
    fig.tight_layout()
    plt.show()


def ensure_groups(records, path):
    if not path.exists():
        # Exact duplicates start together. Repeated arrangements still need human review.
        hashes = {}
        rows = []
        for r in records:
            group = hashes.setdefault(r["image_sha256"], r["stem"])
            rows.append({"image": r["image"], "group": group})
        write_csv(path, rows, ["image", "group"])
    return read_csv(path)


def frozen_split(records, fingerprint, groups_path, output_path, reviewed=False, seed=42):
    rows = read_csv(groups_path)
    if len(rows) != len(records) or {r["image"] for r in rows} != {r["image"] for r in records}:
        raise ValueError("Photo grouping CSV must contain exactly one row per source image.")
    grouping = {r["image"]: r["group"].strip() for r in rows}
    if not all(grouping.values()):
        raise ValueError("Every photo needs a nonempty group.")
    hashes = {}
    for r in records:
        previous = hashes.setdefault(r["image_sha256"], grouping[r["image"]])
        if previous != grouping[r["image"]]:
            raise ValueError("Identical photos must use the same group.")
    signature = digest_json({"fingerprint": fingerprint, "groups": grouping, "seed": seed, "fractions": [0.7, 0.15, 0.15]})
    if output_path.exists():
        result = json.loads(output_path.read_text())
        if result["signature"] != signature:
            raise ValueError("Dataset/groups changed after the split was frozen. Use a new DATASET_NAME and run name; do not mix evaluations.")
        return result
    if not reviewed:
        print("Split not created. Review photo_groups.csv, then set GROUPS_REVIEWED=True and rerun this cell.")
        return None
    groups = sorted(set(grouping.values()))
    if len(groups) < 3:
        raise ValueError("At least three independent arrangement groups are needed for train/val/test.")
    random.Random(seed).shuffle(groups)
    holdout = min(max(1, round(len(groups) * 0.15)), (len(groups) - 1) // 2)
    group_splits = {g: "test" if i < holdout else "val" if i < 2 * holdout else "train" for i, g in enumerate(groups)}
    result = {"signature": signature, "dataset_fingerprint": fingerprint, "seed": seed,
              "photos": [{"image": r["image"], "group": grouping[r["image"]], "split": group_splits[grouping[r["image"]]]} for r in records]}
    write_json(output_path, result)
    return result


def prepare_detector(root, records, split, work):
    target = work / "detector" / split["signature"][:16]
    by_name = {r["image"]: r for r in records}
    for row in split["photos"]:
        record = by_name[row["image"]]
        for kind, src in [("images", root / "images" / row["image"]), ("labels", root / "labels" / f"{record['stem']}.txt")]:
            dest = target / row["split"] / kind / src.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or sha256(dest) != sha256(src):
                shutil.copy2(src, dest)
    config = target / "data.yaml"
    config.write_text(f"path: {json.dumps(str(target))}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n  0: domino\n")
    # Exporters may read the YAML's val split for calibration. Keep calibration strictly in train.
    (target / "calibration.yaml").write_text(f"path: {json.dumps(str(target))}\ntrain: train/images\nval: train/images\nnames:\n  0: domino\n")
    return config


def rectify_tile(rgb, corners, half_size=128):
    """Four corners in source pixels -> horizontal RGB tile, left half then right half."""
    p = np.asarray(corners, dtype=np.float32).reshape(4, 2)
    if not np.isfinite(p).all() or not cv2.isContourConvex(p) or abs(cv2.contourArea(p)) < 4:
        raise ValueError("Cannot rectify a degenerate domino detection.")
    # Positive winding in image coordinates, then make a long edge the top edge.
    signed_area = np.sum(p[:, 0] * np.roll(p[:, 1], -1) - p[:, 1] * np.roll(p[:, 0], -1))
    if signed_area < 0:
        p = p[::-1]
    lengths = np.linalg.norm(np.roll(p, -1, axis=0) - p, axis=1)
    p = np.roll(p, -int(np.argmax(lengths)), axis=0).copy()
    target = np.float32([[0, 0], [2 * half_size - 1, 0], [2 * half_size - 1, half_size - 1], [0, half_size - 1]])
    transform = cv2.getPerspectiveTransform(p, target)
    return cv2.warpPerspective(rgb, transform, (2 * half_size, half_size), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(127, 127, 127))


def tile_halves(tile):
    center = tile.shape[1] // 2
    return tile[:, :center].copy(), tile[:, center:].copy()


def score_pairs(pairs, blank_score=0):
    return sum(blank_score if value == 0 else value for pair in pairs if pair is not None for value in pair)


def make_crops(root, records, directory, fingerprint):
    metadata = {"dataset_fingerprint": fingerprint, "crop_version": CROP_VERSION}
    marker = directory / "metadata.json"
    if marker.exists() and json.loads(marker.read_text()) != metadata:
        raise ValueError("Crop dataset changed. Use a new crop directory so existing pip labels remain meaningful.")
    directory.mkdir(parents=True, exist_ok=True)
    write_json(marker, metadata)
    tiles = []
    for record in records:
        rgb = read_rgb(root / "images" / record["image"])
        for index, box in enumerate(record["boxes"]):
            tile_id = f"{record['stem']}_d{index:04d}"
            paths = {side: directory / "halves" / f"{tile_id}_{side}.png" for side in ("a", "b")}
            if not all(p.is_file() for p in paths.values()):
                tile = rectify_tile(rgb, np.asarray(box) * [record["width"], record["height"]])
                for side, half in zip(("a", "b"), tile_halves(tile)):
                    paths[side].parent.mkdir(parents=True, exist_ok=True)
                    Image.fromarray(half).save(paths[side])
            tiles.append({"tile_id": tile_id, "image": record["image"], "box_index": index,
                          "a_path": str(paths["a"].relative_to(directory)), "b_path": str(paths["b"].relative_to(directory))})
    write_csv(directory / "tiles.csv", tiles, ["tile_id", "image", "box_index", "a_path", "b_path"])
    return tiles


def load_pip_labels(path, tiles):
    fields = ["tile_id", "a", "b", "status"]
    if not path.exists():
        write_csv(path, [{"tile_id": t["tile_id"], "a": "", "b": "", "status": "pending"} for t in tiles], fields)
    rows = read_csv(path)
    if len(rows) != len(tiles) or {r["tile_id"] for r in rows} != {t["tile_id"] for t in tiles}:
        raise ValueError("Pip label IDs do not match the crop manifest.")
    for row in rows:
        if row["status"] not in {"pending", "labeled", "rejected"}:
            raise ValueError(f"Invalid labeling status: {row}")
        if row["status"] == "labeled" and (row["a"] not in CLASS_NAMES or row["b"] not in CLASS_NAMES):
            raise ValueError(f"Both halves must have values 0–15: {row}")
    return {r["tile_id"]: r for r in rows}


def classifier_rows(tiles, labels, split):
    photo_split = {r["image"]: r["split"] for r in split["photos"]}
    result = []
    for tile in tiles:
        label = labels[tile["tile_id"]]
        if label["status"] != "labeled":
            continue
        for side in ("a", "b"):
            result.append({"tile_id": tile["tile_id"], "image": tile["image"], "side": side,
                           "path": tile[f"{side}_path"], "value": int(label[side]), "split": photo_split[tile["image"]]})
    return result


def polygon_iou(a, b):
    a, b = np.asarray(a, np.float32), np.asarray(b, np.float32)
    area_a, area_b = abs(cv2.contourArea(a)), abs(cv2.contourArea(b))
    intersection, _ = cv2.intersectConvexConvex(a, b)
    return max(0.0, float(intersection)) / max(area_a + area_b - intersection, 1e-9)


def match_boxes(truth, predicted, threshold=0.5):
    candidates = sorted([(polygon_iou(a, b), i, j) for i, a in enumerate(truth) for j, b in enumerate(predicted)], reverse=True)
    used_truth, used_pred, matches = set(), set(), []
    for overlap, i, j in candidates:
        if overlap >= threshold and i not in used_truth and j not in used_pred:
            used_truth.add(i)
            used_pred.add(j)
            matches.append((i, j, overlap))
    return matches

## Load both models
Mount the Drive containing `MyDrive/DominoesCalculator/models`. Both checkpoints
and their saved dataset contracts must exist. CPU inference avoids competing GPU use
between PyTorch and TensorFlow. Changing run names requires rerunning this section.

In [ ]:
import tensorflow as tf
from ultralytics import YOLO
from IPython.display import display
from uuid import uuid4

PROJECT, WORK, IN_COLAB = setup_project()
detector_run = PROJECT / "models/detector" / DETECTOR_RUN_NAME
classifier_run = PROJECT / "models/classifier" / CLASSIFIER_RUN_NAME
detector_path = detector_run / "weights/best.pt"
classifier_path = classifier_run / "best.keras"
for path in [detector_path, classifier_path, detector_run / "dataset_contract.json",
             classifier_run / "dataset_contract.json"]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing saved model file: {path}. Check your run names and Drive account.")
detector_contract = json.loads((detector_run / "dataset_contract.json").read_text())
classifier_contract = json.loads((classifier_run / "dataset_contract.json").read_text())
if detector_contract["split_signature"] != classifier_contract["split_signature"]:
    raise ValueError("These models were trained with different photo splits. Choose matching runs.")
if (classifier_contract["crop_version"] != CROP_VERSION
        or classifier_contract["class_names"] != CLASS_NAMES
        or classifier_contract["image_size"] != 128):
    raise ValueError("Classifier crop geometry or class order does not match this notebook.")
detector_model = YOLO(str(detector_path))
with tf.device("/CPU:0"):
    classifier = tf.keras.models.load_model(classifier_path, compile=False)
if tuple(classifier.input_shape[1:]) != (128, 128, 3) or classifier.output_shape[-1] != 16:
    raise ValueError("Expected a 128×128 RGB classifier with 16 outputs.")


def half_predict(half):
    with tf.device("/CPU:0"):
        # RGB 0–255; normalization is already inside the saved classifier.
        inputs = {classifier.inputs[0].name.split(":")[0]: np.asarray(half, np.float32)[None]}
        return classifier(inputs, training=False).numpy()[0]


print("Detector:", detector_path)
print("Pip classifier:", classifier_path)
print("Blank-half score:", BLANK_SCORE)

## Read the photo
Overlapping detections are marked as possible duplicates; they remain in the predicted
total so you can inspect them. Orange outlines also mark low-confidence or unreadable
crops. A missing tile may have no outline or warning, so compare the overlay to the photo.

In [ ]:
def predict_photo(rgb, detector, predict_half, detector_image_size=640,
                  detection_confidence=0.25, pip_confidence=0.8, blank_score=50):
    # YOLO's NumPy input uses BGR; crop geometry and classifier input use RGB.
    result = detector.predict(rgb[:, :, ::-1].copy(), imgsz=detector_image_size,
                              conf=detection_confidence, device="cpu", verbose=False)[0]
    corners = result.obb.xyxyxyxy.cpu().numpy()
    detection_scores = result.obb.conf.cpu().numpy()
    rows, crops, pairs = [], [], []
    for index, box in enumerate(corners):
        row = {"tile": index + 1, "a": None, "b": None, "score": None,
               "detector_confidence": float(detection_scores[index]),
               "a_confidence": None, "b_confidence": None,
               "corners": box.tolist(), "review_reasons": []}
        crop = None
        pair = None
        try:
            crop = rectify_tile(rgb, box)
            probabilities = [np.asarray(predict_half(half)) for half in tile_halves(crop)]
            if any(p.shape != (16,) or not np.isfinite(p).all() for p in probabilities):
                raise ValueError("Classifier returned invalid probabilities.")
            pair = tuple(int(p.argmax()) for p in probabilities)
            row.update(a=pair[0], b=pair[1], score=score_pairs([pair], blank_score),
                       a_confidence=float(probabilities[0].max()),
                       b_confidence=float(probabilities[1].max()))
            if min(row["a_confidence"], row["b_confidence"]) < pip_confidence:
                row["review_reasons"].append("Low pip confidence")
        except ValueError as error:
            row["review_reasons"].append(f"Unreadable: {error}")
        rows.append(row)
        crops.append(crop)
        pairs.append(pair)
    for i, box in enumerate(corners):
        for j in range(i + 1, len(corners)):
            if polygon_iou(box, corners[j]) >= 0.5:
                rows[i]["review_reasons"].append(f"Possible duplicate of tile {j + 1}")
                rows[j]["review_reasons"].append(f"Possible duplicate of tile {i + 1}")
    for row in rows:
        row["needs_review"] = bool(row["review_reasons"])
    complete = bool(rows) and all(pair is not None for pair in pairs)
    return {"detected_tiles": len(rows), "blank_score": blank_score,
            "predicted_total": score_pairs(pairs, blank_score) if complete else None,
            "readable_subtotal": score_pairs(pairs, blank_score),
            "raw_pips": score_pairs(pairs) if complete else None,
            "needs_review": not complete or any(row["needs_review"] for row in rows),
            "tiles": rows}, crops


def show_photo_result(rgb, report, crops, title):
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(rgb)
    ax.axis("off")
    total = report["predicted_total"]
    total_text = f"Predicted total: {total}" if total is not None else "No complete total available"
    ax.set_title(f"{title}\n{total_text} · {report['detected_tiles']} detected tiles · blanks = {report['blank_score']}")
    for row in report["tiles"]:
        points = np.asarray(row["corners"])
        outline = np.vstack([points, points[0]])
        color = "#ffad33" if row["needs_review"] else "#50c9ff"
        ax.plot(outline[:, 0], outline[:, 1], color=color, linewidth=2)
        values = "unreadable" if row["a"] is None else f"{row['a']} | {row['b']}"
        x, y = points.mean(axis=0)
        ax.text(x, y, f"#{row['tile']}  {values}", ha="center", va="center", color="white",
                fontsize=10, bbox={"facecolor": "#18232d", "alpha": 0.85, "edgecolor": color})
    fig.tight_layout()
    plt.show()
    for start in range(0, len(crops), 6):
        count = min(6, len(crops) - start)
        preview, axes = plt.subplots(math.ceil(count / 3), 3, figsize=(15, 3 * math.ceil(count / 3)), squeeze=False)
        for axis in axes.flat:
            axis.axis("off")
        for axis, row, crop in zip(axes.flat, report["tiles"][start:start + 6], crops[start:start + 6]):
            if crop is not None:
                axis.imshow(crop)
            values = "unreadable" if row["a"] is None else f"{row['a']} | {row['b']} · score {row['score']}"
            axis.set_title(f"#{row['tile']} · {values}", fontsize=11)
        preview.tight_layout()
        plt.show()
        plt.close(preview)
    return fig

## Upload and test
Run this cell again for more photos. You may select several JPEG/PNG files at once.
Each result is saved in a new folder under `MyDrive/DominoesCalculator/photo_tests/`:
the oriented input photo, annotated overlay, per-tile CSV, and JSON predictions.
These files do not change your training data, labels, checkpoints, or frozen split.

In [ ]:
if PHOTO_PATH:
    photo_path = Path(PHOTO_PATH).expanduser()
    uploads = {photo_path.name: photo_path.read_bytes()}
elif IN_COLAB:
    from google.colab import files
    uploads = files.upload()
else:
    raise RuntimeError("Set PHOTO_PATH to a JPEG or PNG, then rerun this cell.")

for filename, content in uploads.items():
    if Path(filename).suffix.lower() not in IMAGE_EXTENSIONS:
        print(f"Skipped {filename}: select a JPEG or PNG. Convert HEIC photos before uploading.")
        continue
    with Image.open(io.BytesIO(content)) as photo:
        rgb = np.asarray(ImageOps.exif_transpose(photo).convert("RGB"))
    report, crops = predict_photo(rgb, detector_model, half_predict,
                                 detector_image_size=detector_contract["image_size"],
                                 detection_confidence=DETECTION_CONFIDENCE,
                                 pip_confidence=PIP_CONFIDENCE, blank_score=BLANK_SCORE)
    report.update(photo=Path(filename).name, detector_run=DETECTOR_RUN_NAME,
                  classifier_run=CLASSIFIER_RUN_NAME, detection_confidence=DETECTION_CONFIDENCE,
                  pip_confidence=PIP_CONFIDENCE)
    output_dir = PROJECT / "photo_tests" / uuid4().hex
    output_dir.mkdir(parents=True)
    Image.fromarray(rgb).save(output_dir / "photo.png")
    overlay = show_photo_result(rgb, report, crops, Path(filename).name)
    overlay.savefig(output_dir / "overlay.png", dpi=150, bbox_inches="tight")
    plt.close(overlay)
    write_json(output_dir / "predictions.json", report)
    table = pd.DataFrame([{k: v for k, v in row.items() if k != "corners"} for row in report["tiles"]],
                         columns=["tile", "a", "b", "score", "detector_confidence", "a_confidence",
                                  "b_confidence", "review_reasons", "needs_review"])
    table.to_csv(output_dir / "tiles.csv", index=False)
    display(table)
    if not report["detected_tiles"]:
        print("No dominoes found. No reliable total; check the photo and detection threshold.")
    elif report["predicted_total"] is None:
        print("Some tiles could not be read. Readable subtotal:", report["readable_subtotal"])
    else:
        print("Predicted total:", report["predicted_total"], "· Raw pips:", report["raw_pips"])
    if report["needs_review"]:
        print("Review the orange outlines and the reasons in the table.")
    print("Compare every outline and value to the photo; unflagged predictions can still be wrong.")
    print("Saved photo and results:", output_dir)